<a href="https://colab.research.google.com/github/dokhacduy/DKduy.github.io/blob/main/colab_adam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiên Đoàn TTS — Adam

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [1]:
print('Dang cai dat (~1 phut)...')

!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"

print('Cai dat hoan tat!')

# Upload voice sample
from google.colab import files

uploaded = files.upload()

VOICE_FILE = next(iter(uploaded))

print(f'Da tai voice sample: {VOICE_FILE}')

Dang cai dat (~1 phut)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 42.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.11.1 requires numpy>=2.1, but you have numpy 2.0.2 which is incompatible.
tifffile 2026.8.23 requires numpy>=2.1, but you have numpy 2.0.2 which is incompatible.
dm-tree 0.1.10 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 2.0.2 which is incompatible.
ml-dtypes 0.6.0 require

Saving voice_preview_adam m .mp3 to voice_preview_adam m .mp3
Da tai voice sample: voice_preview_adam m .mp3


In [ ]:
print('Dang khoi dong OmniVoice...')

import logging
import os
import re
import time

import numpy as np
import torch
import gradio as gr

# ============================================================
# CHECK GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        'Khong co GPU. Hay vao Runtime > Change runtime type > chon T4 GPU.'
    )

print(f'GPU: {torch.cuda.get_device_name(0)}')

# ============================================================
# OPTIONAL TORCH PATCH
# ============================================================

import torch as _torch

if not hasattr(_torch, '_utils'):
    try:
        _torch._utils = _torch._C._utils
        print('Da ap dung torch._utils patch')
    except Exception:
        print('Bo qua torch._utils patch')

# ============================================================
# TRANSFORMERS PATCH
# ============================================================

import transformers as _tf

if not hasattr(_tf, 'AutoFeatureExtractor'):

    class _SafeAutoFeatureExtractor:

        @staticmethod
        def from_pretrained(model_name, **kwargs):

            try:
                from transformers import AutoConfig

                cfg = AutoConfig.from_pretrained(
                    model_name,
                    trust_remote_code=True,
                    **kwargs
                )

                sr = getattr(cfg, 'sampling_rate', 24000)

            except Exception:
                sr = 24000

            class _Result:
                sampling_rate = sr

            return _Result()

    _tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

# ============================================================
# IMPORT OMNIVOICE
# ============================================================

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s'
)

logger = logging.getLogger(__name__)

# ============================================================
# LOAD MODEL
# ============================================================

DEVICE = get_best_device()

logger.info(f'Loading OmniVoice on {DEVICE}...')

model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map=DEVICE,
    dtype=torch.float16,
    load_asr=True
)

SAMPLING_RATE = model.sampling_rate

logger.info(
    f'Model ready - SR: {SAMPLING_RATE}Hz'
)

# ============================================================
# VOICE CLONING
# ============================================================

VOICE_PATH = f'/content/{VOICE_FILE}'

if not os.path.exists(VOICE_PATH):
    raise FileNotFoundError(
        f'Khong tim thay voice sample: {VOICE_PATH}'
    )

logger.info(
    f'Creating VoiceClonePrompt from: {VOICE_PATH}'
)

VOICE_PROMPT = model.create_voice_clone_prompt(
    ref_audio=VOICE_PATH
)

logger.info('Voice prompt ready!')

# ============================================================
# GENERATION CONFIG
# ============================================================

GEN_CFG = OmniVoiceGenerationConfig(
    num_step=48,
    guidance_scale=2.0,
    denoise=True,
    preprocess_prompt=True,
    postprocess_output=True,
    position_temperature=4.0,
    class_temperature=0.15,
    pad_duration=0.1,
    fade_duration=0.1,
)

# ============================================================
# GENERATE VOICE
# ============================================================

def generate_voice(text):

    if not text:
        return None

    text = text.strip()

    if not text:
        return None

    paragraphs = [
        p.strip()
        for p in re.split(r'\n\s*\n', text)
        if p.strip()
    ]

    audios = []

    for i, paragraph in enumerate(paragraphs):

        logger.info(
            f'Generating paragraph {i + 1}/{len(paragraphs)}'
        )

        result = model.generate(
            text=paragraph,
            voice_clone_prompt=VOICE_PROMPT,
            language='en',
            speed=1.0,
            generation_config=GEN_CFG
        )

        audio = result[0]

        audios.append(audio)

        if i < len(paragraphs) - 1:
            silence = np.zeros(
                int(SAMPLING_RATE * 0.3),
                dtype=np.float32
            )

            audios.append(silence)

    if not audios:
        return None

    audio = np.concatenate(audios)

    audio = np.clip(audio, -1.0, 1.0)

    waveform = (
        audio * 32767
    ).astype(np.int16)

    return (
        SAMPLING_RATE,
        waveform
    )

# ============================================================
# GRADIO
# ============================================================

CSS = """
.gradio-container {
    max-width: 720px !important;
    margin: 0 auto !important;
    padding: 16px !important;
}

footer {
    display: none !important;
}
"""

THEME = gr.themes.Soft(
    primary_hue='indigo'
)

gr.close_all()

with gr.Blocks(
    title='Kiên Đoàn TTS - Adam',
    theme=THEME,
    css=CSS
) as demo:

    gr.Markdown(
        '# Kiên Đoàn TTS\n'
        'Adam - Giọng nam phù hợp narration, '
        'video ngắn và nội dung YouTube.'
    )

    t = gr.Textbox(
        label='Nhập văn bản',
        lines=8,
        placeholder='Nhập văn bản tiếng Anh...'
    )

    btn = gr.Button(
        'Tạo giọng nói',
        variant='primary'
    )

    out = gr.Audio(
        label='Kết quả',
        type='numpy'
    )

    btn.click(
        generate_voice,
        inputs=t,
        outputs=out,
        concurrency_limit=1
    )

print('Dang khoi dong giao dien...')

demo.launch(
    server_name='0.0.0.0',
    share=True,
    debug=True
)


Dang khoi dong OmniVoice...
GPU: Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

Dang khoi dong giao dien...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2989fa8315d39fac43.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
